# Concurrency: threads, processes and async

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-08-concurrency.ipynb)

A pipeline spends its time in two kinds of work: waiting (network, disk, a database) and computing (parsing, hashing,
aggregating). Python handles them differently. This chapter explains the Global Interpreter Lock, when threads help,
when processes help, and how `asyncio` handles thousands of waits with one thread.

## The GIL in one paragraph

CPython has a **Global Interpreter Lock**: only one thread runs Python bytecode at a time. Threads still help when a
thread spends its time waiting, because the lock is released during I/O. Threads do **not** speed up pure Python
computation. For that, use processes (separate interpreters) or a library that releases the GIL, such as NumPy or
Polars.

Rule of thumb: **threads for waiting, processes for computing.**

## Threads for I/O

`ThreadPoolExecutor` runs a function over many inputs with a pool of threads. It fits fetching many URLs, calling an API
or reading many files. `pool.map` returns results in input order.

The example below simulates a 0.2-second network call. Ten calls done one by one take two seconds; with ten threads they
take about 0.2 seconds.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def fetch(page):
    time.sleep(0.2)           # stands in for a network round trip
    return page * 10

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=10) as pool:
    results = list(pool.map(fetch, range(10)))
elapsed = time.perf_counter() - start

assert results == [n * 10 for n in range(10)]    # order is preserved
assert elapsed < 1.0                             # far below the 2 seconds sequential would take

## Shared state needs a lock

Threads share memory. Two threads that update one counter can interleave and lose an update. A `Lock` makes the
read-modify-write step atomic. Prefer passing results back (as `pool.map` does) over sharing mutable state.

In [ ]:
import threading

counter = {"value": 0}
lock = threading.Lock()

def bump(times):
    for _ in range(times):
        with lock:
            counter["value"] += 1

threads = [threading.Thread(target=bump, args=(10_000,)) for _ in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

assert counter["value"] == 40_000

## Processes for computation

`ProcessPoolExecutor` runs work in separate interpreters, so CPU-bound code uses several cores. The worker function must
be defined in an importable module, because each process loads it by name. In a notebook this works on Linux and Colab,
but not reliably on every platform, so the example below is shown as a reference rather than run:

```python
from concurrent.futures import ProcessPoolExecutor

def checksum(chunk):
    return sum(x * x for x in chunk)

if __name__ == "__main__":
    chunks = [range(i, i + 1_000_000) for i in range(0, 4_000_000, 1_000_000)]
    with ProcessPoolExecutor() as pool:
        partials = list(pool.map(checksum, chunks))
    total = sum(partials)
```

Each process has its own memory, so arguments and results are copied between processes. Send compact inputs, such as
file paths, and return small summaries.

## `asyncio` for many waits

`async def` defines a coroutine. `await` pauses it while it waits for something. One event loop runs thousands of
coroutines that mostly wait, with no threads and no lock.

Notebooks already run an event loop, so you can `await` at the top level of a cell.

In [ ]:
import asyncio

async def call_api(endpoint):
    await asyncio.sleep(0.1)  # stands in for an HTTP request
    return f"{endpoint}: 200"

async def fetch_all(endpoints):
    return await asyncio.gather(*(call_api(e) for e in endpoints))

endpoints = [f"/items/{n}" for n in range(50)]
start = time.perf_counter()
responses = await fetch_all(endpoints)
elapsed = time.perf_counter() - start

assert responses[0] == "/items/0: 200" and len(responses) == 50
assert elapsed < 1.0      # 50 waits of 0.1 s overlapped, not 5 seconds in a row

## Limiting concurrency

Unbounded concurrency can overload the system you call: a database with 500 connections, or an API with a rate limit.
An `asyncio.Semaphore` caps how many coroutines run at once.

In [ ]:
async def bounded_fetch(endpoints, limit):
    gate = asyncio.Semaphore(limit)
    peak = {"now": 0, "max": 0}

    async def one(endpoint):
        async with gate:
            peak["now"] += 1
            peak["max"] = max(peak["max"], peak["now"])
            await asyncio.sleep(0.01)
            peak["now"] -= 1
            return endpoint

    results = await asyncio.gather(*(one(e) for e in endpoints))
    return results, peak["max"]

results, peak = await bounded_fetch(list(range(20)), limit=4)
assert len(results) == 20
assert peak == 4          # never more than four in flight

## Practice questions

1. Explain why `ThreadPoolExecutor` speeds up a loop of HTTP calls but does not speed up a loop that parses numbers.
2. Write `async def first_ok(endpoints)` that calls `call_api` for each endpoint concurrently and returns the endpoints
   whose response ends with `200`.
3. Use `threading.Lock` correctly: write a function that appends to a shared list from several threads, then check
   the length.

## Answers

### Question 1

HTTP calls spend their time waiting, and threads overlap waits. Number parsing is pure Python, so the GIL lets only one
thread run at a time. Use processes for that work.

### Question 2

In [ ]:
async def first_ok(endpoints):
    responses = await asyncio.gather(*(call_api(e) for e in endpoints))
    return [e for e, r in zip(endpoints, responses) if r.endswith("200")]

assert await first_ok(["/a", "/b"]) == ["/a", "/b"]

### Question 3

In [ ]:
shared = []
shared_lock = threading.Lock()

def append_many(n):
    for i in range(n):
        with shared_lock:
            shared.append(i)

workers = [threading.Thread(target=append_many, args=(1_000,)) for _ in range(5)]
for worker in workers:
    worker.start()
for worker in workers:
    worker.join()

assert len(shared) == 5_000

## Interview notes

- **Threads for waiting, processes for computing, async for very many waits.**
- **The GIL is about bytecode, not your data.** NumPy and Polars release it during heavy computation.
- **Bound concurrency.** A semaphore or a fixed pool size protects the downstream system.
- **Shared mutable state needs a lock.** Passing results back through a pool is simpler and safer.